# Solvencia Intertemporal de la Deuda Consolidada Argentina (2004-2025)
## Notebook único de ejecución del proceso empírico completo

Este notebook ejecuta, en el orden correcto de dependencias, la totalidad del proceso empírico que produce los datos, cuadros y figuras citados en la tesis "La solvencia intertemporal de la deuda consolidada argentina post-2025" (FCE-UNCuyo).

Cada celda de código llama a un script real ubicado en `codigo/`, el mismo que se ejecuta de forma independiente y que está documentado y citado en el Capítulo de Metodología y en el Apéndice de la tesis. Este notebook no reimplementa esa lógica: la ejecuta, para que todo el proceso pueda correrse de punta a punta desde un único lugar y quede trazado en una sola salida.

**Estructura del proyecto** (relativa a la raíz `Deuda/`):

| Carpeta | Contenido |
|---|---|
| `codigo/ingesta_datos/` | Descarga y construcción del panel de datos trimestral |
| `codigo/modelos/` | Las 15 etapas de estimación econométrica |
| `codigo/graficos/` | Generación de las figuras de la tesis |
| `datos/` | Panel consolidado (`dataset_consolidado_real.csv`) y datos crudos/procesados |
| `resultados/tablas/` | Salidas tabulares de cada etapa, en CSV |
| `tesis/` | Documento LaTeX y PDF final, con las figuras en `tesis/figuras/` |
| `Bibliografia/` | Fuentes citadas y su verificación (`descargas_verificacion/`) |

**Requisitos**: Python 3.12, y las librerías listadas en `requirements.txt` (pandas, numpy, statsmodels, scipy, arch, matplotlib, seaborn, ruptures, yfinance, requests).

**Nota sobre la Etapa 1 (ingesta)**: las Etapas 1.1 a 1.6 descargan datos desde fuentes externas (BCRA, MECON/INDEC, Yahoo Finance) y requieren conexión a internet; su resultado ya está guardado en `datos/dataset_consolidado_real.csv`, por lo que **pueden omitirse** y correr el notebook directamente desde la Etapa 2 sin perder reproducibilidad de las Etapas 2 en adelante.

In [ ]:
import subprocess
import sys
import pathlib

sys.stdout.reconfigure(encoding="utf-8")
sys.stderr.reconfigure(encoding="utf-8")

RAIZ = pathlib.Path.cwd()
if RAIZ.name == "codigo":
    RAIZ = RAIZ.parent

def ejecutar(ruta_relativa):
    """Ejecuta un script del proyecto como proceso independiente y muestra su
    salida estándar. Cada etapa corre en su propio proceso de Python, con su
    propio directorio de trabajo (la raíz del proyecto), tal como se ejecuta
    de forma individual fuera de este notebook."""
    ruta = RAIZ / ruta_relativa
    resultado = subprocess.run(
        [sys.executable, str(ruta)],
        cwd=RAIZ,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    print(resultado.stdout)
    if resultado.returncode != 0:
        print(resultado.stderr, file=sys.stderr)
        raise RuntimeError(f"{ruta_relativa} terminó con error (código {resultado.returncode})")
    return resultado.stdout

### Etapa 1.1 — Descarga de datos financieros

Descarga el índice de volatilidad VIX (Yahoo Finance) y, en cascada de fuentes de respaldo, el riesgo país EMBI+ argentino.

`codigo/ingesta_datos/ingesta_datos_financieros.py`

In [ ]:
ejecutar("codigo/ingesta_datos/ingesta_datos_financieros.py")

### Etapa 1.2 — Descarga del CER

Descarga el Coeficiente de Estabilización de Referencia desde la API pública del BCRA.

`codigo/ingesta_datos/ingesta_bcra.py`

In [ ]:
ejecutar("codigo/ingesta_datos/ingesta_bcra.py")

### Etapa 1.3 — Descarga de series fiscales y cambiarias

Descarga la deuda pública, el resultado fiscal, el Tipo de Cambio Real Multilateral y el PIB real desde datos.gob.ar / INDEC.

`codigo/ingesta_datos/ingesta_mecon_indec.py`

In [ ]:
ejecutar("codigo/ingesta_datos/ingesta_mecon_indec.py")

### Etapa 1.4 — Construcción del panel consolidado

Orquesta las Etapas 1.1 a 1.3, fusiona las series por trimestre y valida cobertura, escribiendo `datos/dataset_consolidado_real.csv` ($n=88$).

`codigo/ingesta_datos/construccion_dataset.py`

In [ ]:
ejecutar("codigo/ingesta_datos/construccion_dataset.py")

### Etapa 1.5 — Spread soberano regional

Construye el spread soberano regional sobre el ETF EMB (iShares JPMorgan USD Emerging Markets Bond), instrumento del EMBI+ en el IV-2SLS.

`codigo/ingesta_datos/ingesta_spread_regional.py`

In [ ]:
ejecutar("codigo/ingesta_datos/ingesta_spread_regional.py")

### Etapa 1.6 — Pasivos remunerados del BCRA

Descarga LEBAC/NOBAC, LELIQ/NOTALIQ y la posición neta de pases (API BCRA v4.0), insumo de la deuda consolidada SPNF+BCRA.

`codigo/ingesta_datos/ingesta_bcra_pasivos.py`

In [ ]:
ejecutar("codigo/ingesta_datos/ingesta_bcra_pasivos.py")

### Etapa 2.1 — Estacionariedad y quiebre estructural endógeno

Contrastes ADF y KPSS sobre cada serie, complementados con el test de quiebre endógeno de Zivot-Andrews y, como robustez, DF-GLS.

`codigo/modelos/fase1_estacionariedad.py`

In [ ]:
ejecutar("codigo/modelos/fase1_estacionariedad.py")

### Etapa 2.2 — Cointegración de Johansen

Evalúa si el sistema [Deuda/PIB, Resultado Primario, EMBI+, TCRM] admite una relación de equilibrio de largo plazo.

`codigo/modelos/fase2_cointegracion.py`

In [ ]:
ejecutar("codigo/modelos/fase2_cointegracion.py")

### Etapa 2.3 — Función de Reacción Fiscal (DOLS)

Estima la ecuación de Bohn por Mínimos Cuadrados Ordinarios Dinámicos, el estimador principal de la tesis.

`codigo/modelos/fase3_reaccion_fiscal.py`

In [ ]:
ejecutar("codigo/modelos/fase3_reaccion_fiscal.py")

### Etapa 2.4 — Corrección por endogeneidad (IV-2SLS)

Instrumenta el EMBI+ con el VIX y el spread regional para corregir la causalidad inversa entre riesgo soberano y resultado fiscal.

`codigo/modelos/fase4_variables_instrumentales.py`

In [ ]:
ejecutar("codigo/modelos/fase4_variables_instrumentales.py")

### Etapa 2.5 — Modelo de umbral de Hansen (fatiga fiscal)

Testea la hipótesis de fatiga fiscal mediante regresión de umbral con significatividad *bootstrap* del estadístico Sup-LM.

`codigo/modelos/fase5_umbral_hansen.py`

In [ ]:
ejecutar("codigo/modelos/fase5_umbral_hansen.py")

### Etapa 2.6 — Sostenibilidad de la deuda (DSA)

Proyecta la ratio Deuda/PIB a 2035 bajo tres escenarios deterministas y una simulación de Monte Carlo con *shocks* $t$ de Student.

`codigo/modelos/fase6_sostenibilidad_deuda.py`

In [ ]:
ejecutar("codigo/modelos/fase6_sostenibilidad_deuda.py")

### Etapa 2.7 — Diagnósticos de robustez

ACF/PACF, ARCH-LM, CUSUM/CUSUMSQ, Engle-Granger, sensibilidad temporal, causalidad de Granger, filtro de Hamilton y robustez GARCH(1,1) del DSA.

`codigo/modelos/fase7_diagnosticos_robustez.py`

In [ ]:
ejecutar("codigo/modelos/fase7_diagnosticos_robustez.py")

### Etapa 2.8 — Deuda consolidada (SPNF + BCRA)

Suma los pasivos remunerados del BCRA a la deuda del SPNF y reestima la estacionariedad de la serie consolidada.

`codigo/modelos/fase8_deuda_consolidada.py`

In [ ]:
ejecutar("codigo/modelos/fase8_deuda_consolidada.py")

### Etapa 2.9 — Quiebres estructurales múltiples (Bai-Perron)

Identifica quiebres múltiples en la ratio de deuda mediante programación dinámica exacta y selección del número de quiebres por BIC.

`codigo/modelos/fase9_bai_perron.py`

In [ ]:
ejecutar("codigo/modelos/fase9_bai_perron.py")

### Etapa 2.10 — Correlación condicional dinámica (DCC-GARCH)

Reestima por Cuasi-Máxima Verosimilitud la correlación entre los *shocks* del DSA, en reemplazo de la calibración estática.

`codigo/modelos/fase10_dcc_garch.py`

In [ ]:
ejecutar("codigo/modelos/fase10_dcc_garch.py")

### Etapa 2.11 — DOLS por subperíodo

Reestima la Función de Reacción Fiscal en dos submuestras (2004-2014 y 2015-2025) para evaluar estabilidad paramétrica.

`codigo/modelos/fase11_dols_subperiodos.py`

In [ ]:
ejecutar("codigo/modelos/fase11_dols_subperiodos.py")

### Etapa 2.12 — Diagnósticos complementarios del VAR

Selección del orden de rezagos del VAR (AIC/BIC/HQ/FPE), sensibilidad del rango de Johansen, VIF y número de condición.

`codigo/modelos/fase12_diagnosticos_complementarios.py`

In [ ]:
ejecutar("codigo/modelos/fase12_diagnosticos_complementarios.py")

### Etapa 2.13 — Robustez del umbral de Hansen con Δpb_t

Reestima el modelo de umbral con la variable dependiente en primera diferencia (I(0)), que sí satisface el supuesto de estacionariedad del test.

`codigo/modelos/fase13_robustez_hansen_dpb.py`

In [ ]:
ejecutar("codigo/modelos/fase13_robustez_hansen_dpb.py")

### Etapa 2.14 — Robustez del DSA a la dependencia en colas

Reestima la probabilidad de insolvencia bajo marginales $t$ independientes y bajo una cópula Gaussiana, para aislar el efecto de la *tail dependence* de la cópula-$t$.

`codigo/modelos/fase14_dsa_tail_dependence.py`

In [ ]:
ejecutar("codigo/modelos/fase14_dsa_tail_dependence.py")

### Etapa 2.15 — Descomposición ilustrativa del Ajuste Stock-Flujo

Aproxima la descomposición de la identidad de movimiento de la deuda ($SF_t$) para los episodios de 2005, 2018 y 2020.

`codigo/modelos/fase15_sft_decomposicion_ilustrativa.py`

In [ ]:
ejecutar("codigo/modelos/fase15_sft_decomposicion_ilustrativa.py")

### Etapa 3.1 — Figuras principales

Genera las figuras de deuda/resultado primario, dispersión de fatiga fiscal y diagnóstico de primera etapa del IV-2SLS.

`codigo/graficos/generacion_graficos_tesis.py`

In [ ]:
ejecutar("codigo/graficos/generacion_graficos_tesis.py")

### Etapa 3.2 — Cronología de quiebres estructurales

Genera la línea de tiempo de los quiebres de Bai-Perron sobre la serie de deuda.

`codigo/graficos/generacion_cronologia_quiebres.py`

In [ ]:
ejecutar("codigo/graficos/generacion_cronologia_quiebres.py")

## Cierre

Al finalizar todas las etapas, `datos/dataset_consolidado_real.csv` queda actualizado con las columnas derivadas (deuda consolidada), `resultados/tablas/` contiene la totalidad de las salidas tabulares en CSV, y `tesis/figuras/` contiene las figuras referenciadas en el documento. El código fuente íntegro de cada etapa, concatenado en un único archivo de lectura, está disponible en `codigo/codigo_completo_deuda.py`. El documento final se compila desde `tesis/Tesis.tex` (`pdflatex` → `biber` → `pdflatex` × 2).